In [1]:
import warnings
warnings.simplefilter(action='ignore')

import os
import requests
import joblib

import numpy, pandas
import pandas as pd

import matplotlib.pyplot as plt
import scipy.stats as stats

In [2]:
from catboost import CatBoostRegressor ,Pool
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNetCV
from sklearn.linear_model import LassoCV
from sklearn.linear_model import RidgeCV
from sklearn.linear_model import LassoLars
from sklearn.linear_model import Lasso
from sklearn.linear_model import BayesianRidge
from sklearn.linear_model import TweedieRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression

from sklearn.neural_network import MLPRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestCentroid

from sklearn.naive_bayes import GaussianNB
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import BernoulliNB
from sklearn.naive_bayes import CategoricalNB

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor

from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.ensemble import VotingRegressor
from sklearn.ensemble import StackingRegressor

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import accuracy_score

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import SelectFromModel

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_val_score

In [3]:
os.makedirs('data', exist_ok=True)
os.makedirs('model', exist_ok=True)

In [4]:
train = pandas.read_csv('/kaggle/input/zelestra-phase-2-data/Dataset 1.csv')

In [5]:
train = train[['meteorolgicas_em_08_01_gii', 'celulas_ctin08_cc_08_2_ir_cel_2', 'celulas_ctin08_cc_08_2_ir_cel_1', 
               'celulas_ctin08_cc_08_1_ir_cel_2', 'celulas_ctin08_cc_08_1_ir_cel_1', 'meteorolgicas_em_08_01_ghi', 
               'meteorolgicas_em_03_02_gii', 'celulas_ctin03_cc_03_1_ir_cel_1', 'celulas_ctin03_cc_03_2_ir_cel_2', 
               'celulas_ctin03_cc_03_1_ir_cel_2', 'celulas_ctin03_cc_03_2_ir_cel_1', 'meteorolgicas_em_03_02_ghi', 
               'celulas_ctin08_cc_08_1_t_mod', 'celulas_ctin08_cc_08_2_t_mod', 'meteorolgicas_em_08_01_gii_rear', 
               'inversores_ctin08_inv_08_08_p', 'inversores_ctin08_inv_08_08_p_dc', 'inversores_ctin08_inv_08_08_eact_tot', 
               'meteorolgicas_em_03_02_gii_rear', 'celulas_ctin03_cc_03_1_t_mod', 'celulas_ctin03_cc_03_2_t_mod', 
               'inversores_ctin03_inv_03_03_p_dc', 'inversores_ctin03_inv_03_03_p', 'inversores_ctin03_inv_03_03_eact_tot', 
               'celulas_ctin08_cc_08_2_t_amb', 'celulas_ctin08_cc_08_1_t_amb', 'celulas_ctin03_cc_03_2_t_amb', 
               'celulas_ctin03_cc_03_1_t_amb', 'inversores_ctin03_strings_string10_pv_i9', 'ttr_potenciaproducible']]

In [6]:
train, test = train_test_split(train, test_size=0.07, random_state=10)

In [7]:
train.to_csv('data/train_r.csv', index = False)
test.to_csv('data/test_r.csv', index = False)

In [8]:
test

,meteorolgicas_em_08_01_gii,celulas_ctin08_cc_08_2_ir_cel_2,celulas_ctin08_cc_08_2_ir_cel_1,celulas_ctin08_cc_08_1_ir_cel_2,celulas_ctin08_cc_08_1_ir_cel_1,meteorolgicas_em_08_01_ghi,meteorolgicas_em_03_02_gii,celulas_ctin03_cc_03_1_ir_cel_1,celulas_ctin03_cc_03_2_ir_cel_2,celulas_ctin03_cc_03_1_ir_cel_2,...,celulas_ctin03_cc_03_2_t_mod,inversores_ctin03_inv_03_03_p_dc,inversores_ctin03_inv_03_03_p,inversores_ctin03_inv_03_03_eact_tot,celulas_ctin08_cc_08_2_t_amb,celulas_ctin08_cc_08_1_t_amb,celulas_ctin03_cc_03_2_t_amb,celulas_ctin03_cc_03_1_t_amb,inversores_ctin03_strings_string10_pv_i9,ttr_potenciaproducible
12066,452.116883,467.210526,464.095238,464.653846,469.857143,290.704225,390.562500,419.375000,425.000000,411.923077,...,24.770000,1.649478,1.614659,0.401,18.866667,17.370000,19.190000,16.225000,8.843675,24.596833
6647,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,7.950000,0.000000,0.000000,0.000,8.414286,8.542857,9.560000,9.150000,NaN,0.000000
2273,742.864865,760.789474,757.750000,763.440000,768.956522,515.135593,744.179487,764.000000,752.916667,763.714286,...,54.300001,3.061733,3.000100,0.751,37.633334,33.618182,42.183333,36.836364,18.681738,35.435300
2315,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,16.033333,0.000000,0.000000,0.000,15.520000,15.662500,16.700000,16.500000,NaN,0.000000
16979,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,12.900000,0.000000,0.000000,0.000,11.271429,11.244444,14.040000,13.420000,NaN,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4570,608.850000,614.312500,612.000000,617.500000,620.400000,482.216216,628.365385,642.666667,634.400000,641.500000,...,49.533333,2.623352,2.570848,0.641,38.507142,33.500000,39.871429,33.920000,15.146100,30.348271
1981,713.895833,716.941176,714.117647,719.684211,723.500000,627.815789,735.878049,746.923077,740.833333,744.769231,...,51.677778,3.101256,3.039066,0.767,40.418182,37.023077,40.875000,34.437500,18.660977,34.481080
11258,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,7.200000,0.000000,0.000000,0.000,4.987500,5.277778,8.520000,7.900000,NaN,0.000000
999,267.356322,268.833333,265.055556,267.851852,268.560000,191.875000,314.172414,300.312500,289.166667,298.071429,...,25.916666,1.228275,1.204341,0.303,23.200000,22.821428,22.300000,21.925000,8.052000,15.018545


In [9]:
encoder = LabelEncoder()
normalize = SimpleImputer(strategy='mean')

for i in zip(train.columns, train.dtypes):
    if i[1]=='O':
        train[i[0]] = train[i[0]].fillna('Unknown')
        train[i[0]] = normalize.fit_transform(encoder.fit_transform(train[i[0]].to_numpy().reshape(-1,1)).reshape(-1,1))
    else:
        train[i[0]].fillna(train[i[0]].abs().mean(), inplace=True)
        train[i[0]] = normalize.fit_transform(numpy.array(train[i[0]].abs()).reshape(-1,1))
for i in zip(test.columns, test.dtypes):
    if i[1]=='O':
        test[i[0]] = test[i[0]].fillna('Unknown')
        test[i[0]] = normalize.fit_transform(numpy.array(encoder.fit_transform(test[i[0]].to_numpy().reshape(-1,1))).reshape(-1,1))
    else:
        test[i[0]].fillna(test[i[0]].abs().mean(), inplace=True)
        test[i[0]] = normalize.fit_transform(numpy.array(test[i[0]]).reshape(-1,1))

In [10]:
train_c_x, train_c_y = train.drop(columns = ['ttr_potenciaproducible']), train['ttr_potenciaproducible']

train_c_x = train_c_x.reset_index().drop(columns=['index'])
train_c_y = train_c_y.reset_index().drop(columns=['index'])

test_c_x, test_c_y = test.drop(columns = ['ttr_potenciaproducible']), test['ttr_potenciaproducible']

test_c_x = test_c_x.reset_index().drop(columns=['index'])
test_c_y = test_c_y.reset_index().drop(columns=['index'])

In [11]:
train_c_x

,meteorolgicas_em_08_01_gii,celulas_ctin08_cc_08_2_ir_cel_2,celulas_ctin08_cc_08_2_ir_cel_1,celulas_ctin08_cc_08_1_ir_cel_2,celulas_ctin08_cc_08_1_ir_cel_1,meteorolgicas_em_08_01_ghi,meteorolgicas_em_03_02_gii,celulas_ctin03_cc_03_1_ir_cel_1,celulas_ctin03_cc_03_2_ir_cel_2,celulas_ctin03_cc_03_1_ir_cel_2,...,celulas_ctin03_cc_03_1_t_mod,celulas_ctin03_cc_03_2_t_mod,inversores_ctin03_inv_03_03_p_dc,inversores_ctin03_inv_03_03_p,inversores_ctin03_inv_03_03_eact_tot,celulas_ctin08_cc_08_2_t_amb,celulas_ctin08_cc_08_1_t_amb,celulas_ctin03_cc_03_2_t_amb,celulas_ctin03_cc_03_1_t_amb,inversores_ctin03_strings_string10_pv_i9
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,5.066667,5.780000,0.000000,0.000000,0.000,5.328571,4.871429,7.100000,6.400000,7.444889
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,9.100000,9.216667,0.000000,0.000000,0.000,8.050000,8.077778,10.028571,9.487500,7.444889
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,9.250000,9.350000,0.000000,0.000000,0.000,9.250000,9.271429,10.320000,9.922222,7.444889
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,14.040000,14.500000,0.000000,0.000000,0.000,14.700000,15.250000,15.025000,14.671429,7.444889
4,26.647059,24.437500,24.733333,26.315789,25.222222,25.388889,27.235294,25.357143,24.888889,25.307692,...,17.750000,17.900000,0.103341,0.098711,0.024,17.240000,17.216667,17.933334,17.614286,0.961769
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16243,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,12.240000,12.066667,0.000000,0.000000,0.000,9.457143,9.700000,13.833333,13.333333,0.259750
16244,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.675000,0.800000,0.000000,0.000000,0.000,2.300000,2.236364,1.180000,0.442857,7.444889
16245,181.028571,173.315789,174.333333,165.034483,167.200000,165.800000,169.835821,149.071429,148.272727,159.285714,...,32.387499,32.111111,0.614220,0.598044,0.156,27.206667,25.050000,24.933334,22.945455,4.433531
16246,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,5.725000,5.566667,0.000000,0.000000,0.000,3.528571,3.850000,7.380000,6.700000,7.444889


In [12]:
class solo_models:
    def __new__(self, train_c_x, train_c_y, test_c_x, test_c_y):
        print('INITIATING PARAMETER')
        self.X_train, self.X_test, self.y_train, self.y_test = train_c_x, test_c_x, train_c_y, test_c_y
        #print("train_c_x:",self.train_c_x.index,"train_c_y:",self.train_c_y.index,"X_tain:",self.X_train.index)
        self.R_Forest_parm = {'n_estimators' : 25, 
                              'min_samples_split' : 2, 
                              'max_depth' : 10, 
                              'min_samples_leaf' : 2, 
                              'random_state' : 42}
        
        self.Extra_parm = {'n_estimators' : 50, 
                           'min_samples_split' : 2, 
                           'max_depth' : 8, 
                           'min_samples_leaf' : 2, 
                           'random_state' : 42}
        
        self.LGBM_R_parm = {'colsample_bytree': 0.6657998165699125,
                           'drop_rate': 0.8303473371870002,
                           'learning_rate': 0.2762739125344964,
                           'max_bin': 9983,
                           'max_depth': 8623,
                           'max_drop': 5480,
                           'min_child_samples': 101,
                           'min_data_in_leaf': 426,
                           'n_estimators': 1628,
                           'num_leaves': 3640,
                           'objective': 'regression_l1',
                           'reg_alpha': 0.39940189926691194,
                           'reg_lambda': 0.9567353299218986,
                           'skip_drop': 0.6274640597528743,
                           'verbosity': -1}
        
        self.XGB_R_parm = {'colsample_bytree' : 0.871893537724492,
                           'gamma' : 1,
                           'learning_rate' : 0.923192518624813,
                           'max_depth' : 15,
                           'min_child_weight' : 1,
                           'n_estimators' : 17748,
                           'reg_alpha' : 45,
                           'reg_lambda' : 0.8598696247943665,
                           'subsample' : 0.9109367356405415,
                           'random_state' : 42}

        self.catboost_params = {'iterations' : 3000,
                                'learning_rate': 0.009, 
                                'depth': 5, 
                                'l2_leaf_reg': 5.5,
                                'min_child_samples' : 102,
                                'od_wait' : 50,
                                'random_state' : 42,
                                'eval_metric': 'RMSE', 
                                'od_type' : 'Iter',
                                'bootstrap_type': 'Bayesian', 
                                'grow_policy' : 'Depthwise',
                                'logging_level' : 'Silent'}
        
        self.DecisionTree = {'max_depth': 6}
        self.settings = {"time_budget": 2000,
                         "metric": "mae",
                         "estimator_list": ["lgbm"],
                         "task": "regression",
                         "log_file_name": "experiment.log",
                         "seed": 41,
                         "eval_method": "cv"
                    }
        self.X_train.to_csv('data/X_train.csv', index = False)
        self.y_train.to_csv('data/y_train.csv', index = False)
        self.X_test.to_csv('data/X_test.csv', index = False)
        self.y_test.to_csv('data/y_test.csv', index = False)
        
        self.models(self)
        self.manual_training(self)
        return {'manual_ensemble' : self.model_collecter, 'stack_ensemble' : self.stack_training(self)}
        
    def models(self):
        print('LOADING MODEL')
        self.model_collecter = {}
        
        self.LGBM_R = LGBMRegressor(**self.LGBM_R_parm)
        self.XGB_R = XGBRegressor(**self.XGB_R_parm)
        self.catboost = CatBoostRegressor(**self.catboost_params)

        self.model_collecter['random_forest'] = BaggingRegressor(estimator = RandomForestRegressor(**self.R_Forest_parm), 
                                                                 n_estimators = 10, random_state = 0)
        self.model_collecter['extra_trees'] = BaggingRegressor(estimator = ExtraTreesRegressor(**self.Extra_parm), 
                                                                 n_estimators = 10, random_state = 0)
        self.model_collecter['GradientBoostingRegressor'] = BaggingRegressor(estimator = GradientBoostingRegressor(learning_rate=0.1, 
                                                                                                                   min_samples_split=500,
                                                                                                                   min_samples_leaf=50,
                                                                                                                   max_depth=8,
                                                                                                                   max_features='sqrt',
                                                                                                                   subsample=0.8,
                                                                                                                   random_state=10), 
                                                                            n_estimators = 10, random_state = 0)
        
        self.model_collecter['DecisionTreeRegressor'] = BaggingRegressor(estimator = DecisionTreeRegressor(**self.DecisionTree), 
                                                                 n_estimators = 10, random_state = 0)
        self.model_collecter['TweedieRegressor'] = BaggingRegressor(estimator = TweedieRegressor(), 
                                                                 n_estimators = 10, random_state = 0)
        self.model_collecter['LinearRegression'] = BaggingRegressor(estimator = LinearRegression(), 
                                                                 n_estimators = 10, random_state = 0)
        
        self.model_collecter['ElasticNet'] = BaggingRegressor(estimator = ElasticNetCV(), n_estimators = 10, random_state = 0)
        self.model_collecter['LassoCV'] = BaggingRegressor(estimator = LassoCV(), n_estimators = 10, random_state = 0)
        self.model_collecter['RidgeCV'] = BaggingRegressor(estimator = RidgeCV(), n_estimators = 10, random_state = 0)
        self.model_collecter['LassoLars'] = BaggingRegressor(estimator = LassoLars(), n_estimators = 10, random_state = 0)
        self.model_collecter['Lasso'] = BaggingRegressor(estimator = Lasso(), n_estimators = 10, random_state = 0)
        self.model_collecter['BayesianRidge'] = BaggingRegressor(estimator = BayesianRidge(), n_estimators = 10, random_state = 0)
        self.model_collecter['SVR'] = BaggingRegressor(estimator = SVR(), n_estimators = 10, random_state = 0)
        self.model_collecter['PLSRegression'] = BaggingRegressor(estimator = PLSRegression(), n_estimators = 10, random_state = 0)
        self.model_collecter['KMeans'] = BaggingRegressor(estimator = KMeans(), n_estimators = 10, random_state = 0)
        #self.model_collecter['GaussianProcessRegressor'] = GaussianProcessRegressor()
        self.model_collecter['MLPRegressor'] = BaggingRegressor(estimator = MLPRegressor(), n_estimators = 10, random_state = 0)
        
    def manual_training(self):
        print('TRAINING')
        
        for i in self.model_collecter.keys():
            print(f'--{i.upper()}')
            #print(self.X_train)
            self.model_collecter[i].fit(self.X_train, self.y_train)
            joblib.dump(self.model_collecter[i], f'model/{i}.pkl')

        print('--CATBOOST')
        cat_features = list(self.X_train.select_dtypes(include=['object', 'category']).columns)
        train_pool = Pool(self.X_train, self.y_train, cat_features=cat_features)
        val_pool = Pool(self.X_test, self.y_test, cat_features=cat_features)
        
        self.catboost.fit(train_pool, eval_set=(val_pool), verbose=100, early_stopping_rounds=100)
        print('--XGBOOST')
        self.XGB_R.fit(self.X_train, self.y_train, eval_set = [(self.X_test, self.y_test)],eval_metric = "rmse", verbose = False)
        print('--LGBOOST')
        self.LGBM_R.fit(self.X_train, self.y_train,eval_set = [(self.X_test, self.y_test)],eval_metric ='rmse')
        
        self.model_collecter['LGBMRegressor'] = self.LGBM_R
        self.model_collecter['XGBRegressor'] = self.XGB_R
        self.model_collecter['CatBoostRegressor'] = self.catboost

        joblib.dump(self.model_collecter['LGBMRegressor'], f'model/LGBMRegressor.pkl')
        joblib.dump(self.model_collecter['XGBRegressor'], f'model/XGBRegressor.pkl')
        joblib.dump(self.model_collecter['CatBoostRegressor'], f'model/CatBoostRegressor.pkl')
        
    def stack_training(self):    
        print('--STACK')
        #self.model_collecter['LGBMRegressor'] = self.LGBM_R
        #self.model_collecter['XGBRegressor'] = self.XGB_R
        #self.model_collecter['CatBoostRegressor'] = self.catboost
        self.stack_model = ['LGBMRegressor',
                            'LGBMRegressor',
                            'CatBoostRegressor',
                            'random_forest',
                            'extra_trees',
                            'GradientBoostingRegressor',
                            'DecisionTreeRegressor']
        
        estimators = [(i, self.model_collecter[i]) for i in self.model_collecter.keys() if i in self.stack_model]
        self.model = StackingRegressor(estimators, final_estimator = RidgeCV())
        self.model.fit(self.X_train, self.y_train)
        joblib.dump(self.model, f'model/STACK.pkl')
        return self.model

In [13]:
E_model = solo_models(train_c_x, train_c_y, test_c_x, test_c_y)

INITIATING PARAMETER
LOADING MODEL
TRAINING
--RANDOM_FOREST
--EXTRA_TREES
--GRADIENTBOOSTINGREGRESSOR
--DECISIONTREEREGRESSOR
--TWEEDIEREGRESSOR
--LINEARREGRESSION
--ELASTICNET
--LASSOCV
--RIDGECV
--LASSOLARS
--LASSO
--BAYESIANRIDGE
--SVR
--PLSREGRESSION
--KMEANS
--MLPREGRESSOR
--CATBOOST
--XGBOOST
--LGBOOST
--STACK
